In [ ]:
from hcc_release_paths import project_path as _hcc_path
# ============================================================
# Step12.01
# WGCNA project initialization
# ============================================================

import os
import pandas as pd
import numpy as np

project_dir = _hcc_path('')

step12_dir = os.path.join(
    project_dir,
    "LIHC_RNA_Step12_WGCNA"
)

input_dir = os.path.join(
    step12_dir,
    "Input"
)

result_dir = os.path.join(
    step12_dir,
    "Results"
)

fig_dir = os.path.join(
    step12_dir,
    "Figures"
)

for folder in [
    step12_dir,
    input_dir,
    result_dir,
    fig_dir
]:
    os.makedirs(
        folder,
        exist_ok=True
    )


# ------------------------------------------------------------
# 读取365例肿瘤表达矩阵
# Gene × Sample
# ------------------------------------------------------------

expr_file = os.path.join(
    project_dir,
    "LIHC_RNA_Step11_Immune",
    "Input",
    "LIHC_Expression_365_log2CPM.tsv"
)

expr = pd.read_csv(
    expr_file,
    sep="\t",
    index_col=0
)


# ------------------------------------------------------------
# 读取RiskScore和生存资料
# ------------------------------------------------------------

risk_file = os.path.join(
    project_dir,
    "LIHC_RNA_Step11_Immune",
    "Input",
    "LIHC_RiskGroup_365.csv"
)

risk = pd.read_csv(
    risk_file,
    index_col=0
)


expr.index = expr.index.astype(str)
expr.columns = expr.columns.astype(str)
risk.index = risk.index.astype(str)


print("Expression matrix:", expr.shape)
print("Risk table:", risk.shape)

print("\nDuplicated genes:",
      expr.index.duplicated().sum())

print("Missing expression values:",
      expr.isna().sum().sum())

print("\nCommon samples:",
      len(expr.columns.intersection(risk.index)))

print("\nRisk columns:")
print(risk.columns.tolist())

In [ ]:
# ============================================================
# Step12.02
# Remove very low-expression genes
# ============================================================

min_sample_fraction = 0.20
expression_threshold = 1.0

min_samples = int(
    np.ceil(
        expr.shape[1]
        *
        min_sample_fraction
    )
)


expressed_mask = (
    expr > expression_threshold
).sum(axis=1) >= min_samples


expr_expressed = expr.loc[
    expressed_mask
].copy()


print("Original genes:",
      expr.shape[0])

print("Required samples:",
      min_samples)

print("Genes after expression filtering:",
      expr_expressed.shape[0])

print("Samples:",
      expr_expressed.shape[1])

In [ ]:
# ============================================================
# Step12.03
# Select highly variable genes using MAD
# ============================================================

gene_median = expr_expressed.median(
    axis=1
)

gene_mad = expr_expressed.sub(
    gene_median,
    axis=0
).abs().median(
    axis=1
)


mad_table = pd.DataFrame({
    "Gene": expr_expressed.index,
    "MAD": gene_mad.values
})


mad_table = (
    mad_table
    .sort_values(
        "MAD",
        ascending=False
    )
    .reset_index(drop=True)
)


display(
    mad_table.head(20)
)

In [ ]:
# ============================================================
# Select Top 5000 MAD genes
# ============================================================

n_top_genes = 5000


top_genes = (
    mad_table
    .head(n_top_genes)["Gene"]
    .tolist()
)


expr_wgcna = expr_expressed.loc[
    top_genes
].copy()


print(
    "Final WGCNA expression matrix:",
    expr_wgcna.shape
)

In [ ]:
# ============================================================
# Step12.04
# Save WGCNA input
# ============================================================

expr_wgcna.to_csv(
    os.path.join(
        input_dir,
        "LIHC_WGCNA_Top5000_MAD_Genes.tsv"
    ),
    sep="\t"
)


mad_table.to_csv(
    os.path.join(
        result_dir,
        "LIHC_Gene_MAD_Ranking.csv"
    ),
    index=False
)


print("Saved WGCNA input.")

print(
    os.path.join(
        input_dir,
        "LIHC_WGCNA_Top5000_MAD_Genes.tsv"
    )
)

In [ ]:
# ============================================================
# Step12.05
# MAD distribution
# ============================================================

import matplotlib.pyplot as plt


fig, ax = plt.subplots(
    figsize=(7, 5)
)


ax.hist(
    mad_table["MAD"],
    bins=60
)


cutoff_mad = mad_table.iloc[
    n_top_genes - 1
]["MAD"]


ax.axvline(
    cutoff_mad,
    linestyle="--",
    linewidth=1.5
)


ax.set_xlabel(
    "Median absolute deviation (MAD)"
)

ax.set_ylabel(
    "Number of genes"
)

ax.set_title(
    "Gene variability distribution"
)


ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)


fig.tight_layout()


fig.savefig(
    os.path.join(
        fig_dir,
        "WGCNA_Gene_MAD_Distribution.png"
    ),
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    os.path.join(
        fig_dir,
        "WGCNA_Gene_MAD_Distribution.pdf"
    ),
    bbox_inches="tight"
)


plt.show()
plt.close(fig)


print(
    "Top5000 MAD cutoff:",
    cutoff_mad
)

In [ ]:
# ============================================================
# Step12.06
# Sample clustering for outlier detection
# ============================================================

from scipy.cluster.hierarchy import (
    linkage,
    dendrogram
)
from scipy.spatial.distance import pdist


# 转置
# Sample × Gene
datExpr = expr_wgcna.T.copy()


print(
    "WGCNA datExpr:",
    datExpr.shape
)


# Euclidean distance
sample_distance = pdist(
    datExpr,
    metric="euclidean"
)


sample_linkage = linkage(
    sample_distance,
    method="average"
)


fig, ax = plt.subplots(
    figsize=(18, 7)
)


dendrogram(
    sample_linkage,
    labels=datExpr.index,
    leaf_rotation=90,
    leaf_font_size=4,
    ax=ax
)


ax.set_title(
    "Sample clustering for WGCNA quality control",
    fontsize=15
)

ax.set_xlabel(
    "TCGA-LIHC samples"
)

ax.set_ylabel(
    "Clustering distance"
)


fig.tight_layout()


fig.savefig(
    os.path.join(
        fig_dir,
        "WGCNA_Sample_Clustering.png"
    ),
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    os.path.join(
        fig_dir,
        "WGCNA_Sample_Clustering.pdf"
    ),
    bbox_inches="tight"
)


plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# Step12.07
# Quantitative sample outlier detection
# Standardized sample connectivity
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os


# datExpr:
# rows = samples
# columns = Top5000 genes

print("datExpr:", datExpr.shape)


# ------------------------------------------------------------
# Sample × Sample Pearson correlation
# ------------------------------------------------------------

sample_cor = np.corrcoef(
    datExpr.values
)


# Signed adjacency
sample_adj = (
    (1 + sample_cor) / 2
) ** 2


# 去掉自己和自己的连接
np.fill_diagonal(
    sample_adj,
    0
)


# connectivity
sample_k = sample_adj.sum(
    axis=1
)


# standardized connectivity
sample_zk = (
    sample_k - sample_k.mean()
) / sample_k.std(ddof=1)


sample_qc = pd.DataFrame({
    "Sample_ID": datExpr.index,
    "Connectivity": sample_k,
    "Z_connectivity": sample_zk
})


sample_qc = (
    sample_qc
    .sort_values("Z_connectivity")
    .reset_index(drop=True)
)


display(
    sample_qc.head(20)
)

In [ ]:
sample_qc.to_csv(
    os.path.join(
        result_dir,
        "LIHC_WGCNA_Sample_Connectivity_QC.csv"
    ),
    index=False
)

In [ ]:
# ============================================================
# Step12.08
# Identify potential outlier samples
# ============================================================

outlier_threshold = -2.5


potential_outliers = sample_qc.loc[
    sample_qc["Z_connectivity"] < outlier_threshold
].copy()


print(
    "Potential outliers:",
    len(potential_outliers)
)

display(
    potential_outliers
)

In [ ]:
# ============================================================
# Sample connectivity QC plot
# ============================================================

plot_qc = (
    sample_qc
    .sort_values("Z_connectivity")
    .reset_index(drop=True)
)


fig, ax = plt.subplots(
    figsize=(12, 5)
)


ax.scatter(
    np.arange(len(plot_qc)),
    plot_qc["Z_connectivity"],
    s=25
)


ax.axhline(
    -2.5,
    linestyle="--",
    linewidth=1.5
)


ax.axhline(
    -3.0,
    linestyle=":",
    linewidth=1.5
)


ax.set_xlabel(
    "Samples ordered by connectivity"
)

ax.set_ylabel(
    "Standardized connectivity (Z.K)"
)

ax.set_title(
    "Sample connectivity for WGCNA quality control"
)


ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)


fig.tight_layout()


fig.savefig(
    os.path.join(
        fig_dir,
        "WGCNA_Sample_Connectivity_QC.png"
    ),
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    os.path.join(
        fig_dir,
        "WGCNA_Sample_Connectivity_QC.pdf"
    ),
    bbox_inches="tight"
)


plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# Step12.08b
# Outlier clinical inspection
# ============================================================


outlier_samples = (
    potential_outliers["Sample_ID"]
    .tolist()
)


outlier_clinical = risk.loc[
    risk.index.isin(outlier_samples)
].copy()


display(
    outlier_clinical
)

In [ ]:
# ============================================================
# Step12.09.1
# Prepare WGCNA matrix
# ============================================================

import numpy as np
import pandas as pd


print(
    datExpr.shape
)

display(
    datExpr.iloc[:5,:5]
)

In [ ]:
# ============================================================
# Step12.09.2
# Gene correlation matrix
# ============================================================

gene_corr = np.corrcoef(
    datExpr.values,
    rowvar=False
)


print(
    gene_corr.shape
)

print(
    "min:",
    gene_corr.min()
)

print(
    "max:",
    gene_corr.max()
)

In [ ]:
# ============================================================
# Step12.09.3
# Soft threshold testing
# ============================================================

from scipy.stats import linregress


powers = list(range(1,11)) + [
    12,14,16,18,20
]


soft_results = []


for beta in powers:

    print(
        "Testing beta:",
        beta
    )


    # signed adjacency
    adj = (
        (1 + gene_corr) / 2
    ) ** beta


    np.fill_diagonal(
        adj,
        0
    )


    # connectivity
    k = adj.sum(
        axis=0
    )


    # remove zero
    k = k[k>0]


    hist, edges = np.histogram(
        k,
        bins=50
    )


    centers = (
        edges[:-1]
        +
        edges[1:]
    ) / 2


    prob = hist / hist.sum()


    valid = (
        (centers>0)
        &
        (prob>0)
    )


    if valid.sum()>2:


        slope, intercept, r, p, se = linregress(
            np.log10(centers[valid]),
            np.log10(prob[valid])
        )


        r2 = r*r

        signed_r2 = (
            -np.sign(slope)
            *
            r2
        )

    else:

        signed_r2=np.nan


    soft_results.append({

        "Beta":beta,

        "Scale_free_R2":
        signed_r2,

        "Mean_connectivity":
        np.mean(k),

        "Median_connectivity":
        np.median(k),

        "Max_connectivity":
        np.max(k)

    })


soft_table = pd.DataFrame(
    soft_results
)


display(
    soft_table
)

In [ ]:
soft_table.to_csv(
    os.path.join(
        result_dir,
        "LIHC_WGCNA_SoftThreshold_results.csv"
    ),
    index=False
)


print("saved")

In [ ]:
import matplotlib.pyplot as plt


fig, ax = plt.subplots(
    figsize=(7,5)
)


ax.plot(
    soft_table["Beta"],
    soft_table["Scale_free_R2"],
    marker="o"
)


ax.axhline(
    0.8,
    linestyle="--"
)


ax.set_xlabel(
    "Soft threshold power β"
)

ax.set_ylabel(
    "Scale-free topology fit R²"
)


ax.set_title(
    "WGCNA soft threshold selection"
)


ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)


plt.show()

In [ ]:
fig, ax = plt.subplots(
    figsize=(7,5)
)


ax.plot(
    soft_table["Beta"],
    soft_table["Mean_connectivity"],
    marker="o"
)


ax.set_xlabel(
    "Soft threshold power β"
)


ax.set_ylabel(
    "Mean connectivity"
)


ax.set_title(
    "Mean connectivity analysis"
)


ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)


plt.show()

In [ ]:
# ============================================================
# Step12.10.1
# WGCNA parameters
# ============================================================

soft_power = 14

print(
    "Selected soft power:",
    soft_power
)

print(
    "Expression matrix:",
    datExpr.shape
)

In [ ]:
# ============================================================
# Step12.10.2
# adjacency matrix
# ============================================================


adjacency = (
    (1 + gene_corr) / 2
) ** soft_power


np.fill_diagonal(
    adjacency,
    0
)


print(
    adjacency.shape
)


print(
    "Adjacency summary:"
)

print(
    "min:",
    adjacency.min()
)

print(
    "max:",
    adjacency.max()
)

print(
    "mean:",
    adjacency.mean()
)

In [ ]:
np.save(
    os.path.join(
        result_dir,
        "LIHC_WGCNA_Adjacency_beta14.npy"
    ),
    adjacency
)


print("Adjacency saved")

In [ ]:
# ============================================================
# Step12.10.4
# TOM similarity
# ============================================================


import time

start=time.time()


k = adjacency.sum(axis=1)


# shared neighbors
L = adjacency @ adjacency


tom = (
    L + adjacency
) / (
    np.minimum.outer(k,k)
    +1
    - adjacency
)


np.fill_diagonal(
    tom,
    1
)


print(
    "TOM shape:",
    tom.shape
)


print(
    "Time:",
    time.time()-start,
    "seconds"
)

In [ ]:
np.save(
    os.path.join(
        result_dir,
        "LIHC_WGCNA_TOM_beta14.npy"
    ),
    tom
)


print("TOM saved")

In [ ]:
# ============================================================
# Step12.10.6
# TOM distance
# ============================================================


tom_dist = 1 - tom


print(
    tom_dist.shape
)

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram


# condensed distance
from scipy.spatial.distance import squareform


gene_tree = linkage(
    squareform(
        tom_dist,
        checks=False
    ),
    method="average"
)


print(
    "Gene tree generated"
)

In [ ]:
# ==========================================================
# Step12.11.1
# Prepare TOM dissimilarity
# ==========================================================

dissTOM = tom_dist.copy()


print(
    "dissTOM shape:",
    dissTOM.shape
)

print(
    "range:",
    dissTOM.min(),
    dissTOM.max()
)

In [ ]:
import inspect
# ==========================================================
# Step12.11.2
# Dynamic Tree Cut (Python version)
# ==========================================================

from dynamicTreeCut import cutreeHybrid


modules = cutreeHybrid(
    link=gene_tree,
    distM=dissTOM,
    deepSplit=2,
    pamStage=True,
    pamRespectsDendro=True,
    minClusterSize=50,
    verbose=2
)


module_labels = modules["labels"]


print(
    "Number of modules:",
    len(np.unique(module_labels))
)


print(
    np.unique(
        module_labels,
        return_counts=True
    )
)

In [ ]:
# ==========================================================
# Step12.11.3
# Convert module labels to WGCNA colors
# ==========================================================

import pandas as pd


# 查看模块编号
module_count = pd.Series(
    module_labels
).value_counts()


display(
    module_count
)

In [ ]:
# ==========================================================
# Step12.11.3
# Convert module labels to WGCNA colors
# ==========================================================

import pandas as pd


module_labels = np.array(module_labels)


module_color_map = {

    0:"grey",

    1:"turquoise",

    2:"blue",

    3:"brown",

    4:"yellow",

    5:"green",

    6:"red"

}


module_colors = [
    module_color_map[m]
    for m in module_labels
]


module_summary = pd.DataFrame({
    "Module": module_colors
})


display(
    module_summary["Module"]
    .value_counts()
)

In [ ]:
# ==========================================================
# Save module assignment
# ==========================================================


module_assignment = pd.DataFrame({

    "Gene": datExpr.columns,

    "Module": module_colors

})


module_assignment.to_csv(
    os.path.join(
        result_dir,
        "LIHC_WGCNA_Module_Assignment.csv"
    ),
    index=False
)


print(
    "Module assignment saved"
)

In [ ]:
# ==========================================================
# Step12.12 Fixed
# WGCNA dendrogram + module colors
# ==========================================================

import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import dendrogram
import matplotlib.colors as mcolors
import numpy as np


# 获取树排序

dendro = dendrogram(
    gene_tree,
    no_plot=True
)


gene_order = dendro["leaves"]


# 按树顺序排列模块颜色

ordered_colors = [
    module_colors[i]
    for i in gene_order
]


# 转RGB

color_rgb = np.array(
    [
        mcolors.to_rgb(c)
        for c in ordered_colors
    ]
)



fig, axes = plt.subplots(
    2,
    1,
    figsize=(18,10),
    gridspec_kw={
        "height_ratios":[4,0.4]
    }
)



# ==========================
# Gene dendrogram
# ==========================

dendrogram(
    gene_tree,
    no_labels=True,
    color_threshold=None,
    ax=axes[0]
)


axes[0].set_title(
    "Gene dendrogram and module assignment",
    fontsize=16
)


axes[0].set_ylabel(
    "Height"
)



# ==========================
# Module color bar
# ==========================


axes[1].imshow(
    [color_rgb],
    aspect="auto"
)


axes[1].set_yticks([])

axes[1].set_xlabel(
    "Genes"
)


axes[1].set_title(
    "Module colors"
)


plt.tight_layout()

plt.show()

In [ ]:
# ==========================================================
# Restore WGCNA module assignment
# ==========================================================

import pandas as pd
import numpy as np
import os


module_file = (
    r".\LIHC_RNA_Step12_WGCNA\Results\LIHC_WGCNA_Module_Assignment.csv"
)


module_assignment = pd.read_csv(
    module_file
)


print(
    module_assignment.head()
)


print(
    module_assignment.shape
)

In [ ]:
module_colors = (
    module_assignment["Module"]
    .values
)


print(
    len(module_colors)
)


print(
    pd.Series(module_colors).value_counts()
)

In [ ]:
# ==========================================================
# Restore WGCNA expression matrix
# ==========================================================

import pandas as pd
import numpy as np


expr_file = (
    r".\LIHC_RNA_Step11_Immune\Input\LIHC_Expression_365_log2CPM.tsv"
)


expression_matrix = pd.read_csv(
    expr_file,
    sep="\t",
    index_col=0
)


print(expression_matrix.shape)

print(expression_matrix.iloc[:5,:5])

In [ ]:
module_file = (
    r".\LIHC_RNA_Step12_WGCNA\Results\LIHC_WGCNA_Module_Assignment.csv"
)


module_assignment = pd.read_csv(
    module_file
)


print(module_assignment.head())

print(module_assignment.shape)

In [ ]:
top_genes = (
    module_assignment["Gene"]
    .tolist()
)


expression_top5000 = (
    expression_matrix
    .loc[top_genes]
)


print(
    expression_top5000.shape
)

In [ ]:
module_colors = (
    module_assignment["Module"]
    .values
)

In [ ]:
from sklearn.decomposition import PCA


ME_dict={}


for module in sorted(
    set(module_colors)
):

    if module=="grey":
        continue


    idx=np.where(
        module_colors==module
    )[0]


    module_expr = expression_top5000.iloc[
        idx,
        :
    ]


    pca=PCA(
        n_components=1
    )


    ME = pca.fit_transform(
        module_expr.T
    )


    ME_dict[
        "ME_"+module
    ] = ME[:,0]


ME_matrix = pd.DataFrame(
    ME_dict,
    index=expression_top5000.columns
)


print(
    ME_matrix.shape
)


ME_matrix.head()

In [ ]:
# ==========================================================
# Step12.13
# Calculate Module Eigengenes
# ==========================================================

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA


# module颜色

module_colors = (
    module_assignment["Module"]
    .values
)


ME_dict = {}


for module in sorted(
    np.unique(module_colors)
):

    # 去掉灰色模块
    if module == "grey":
        continue


    # 找该模块基因位置

    gene_index = np.where(
        module_colors == module
    )[0]


    module_expression = (
        expression_top5000
        .iloc[
            gene_index,
            :
        ]
    )


    # PCA
    pca = PCA(
        n_components=1
    )


    eigengene = pca.fit_transform(
        module_expression.T
    )


    ME_dict[
        "ME_" + module
    ] = eigengene[:,0]



ME_matrix = pd.DataFrame(
    ME_dict,
    index=expression_top5000.columns
)


print(
    "ME matrix:",
    ME_matrix.shape
)


ME_matrix.head()

In [ ]:
ME_matrix.to_csv(
    r".\LIHC_RNA_Step12_WGCNA\Results\LIHC_WGCNA_Module_Eigengenes.csv"
)


print("Saved")

In [ ]:
ME_zscore = (
    ME_matrix
    .apply(
        lambda x:(x-x.mean())/x.std()
    )
)

In [ ]:
# ==========================================================
# Reload final RiskScore
# ==========================================================

risk_file = (
    r".\LIHC_RNA_Step8_Final_Model\Results\Final_Top10_RiskScore.csv"
)


risk_df = pd.read_csv(
    risk_file,
    index_col=0
)


print(risk_df.shape)

print(
    risk_df.columns.tolist()
)

risk_df.head()

In [ ]:
ME_risk = ME_matrix.join(
    risk_df[
        [
            "RiskScore"
        ]
    ],
    how="inner"
)


print(ME_risk.shape)

ME_risk.head()

In [ ]:
# ==========================================================
# Step12.14.1
# Merge WGCNA ME with final RiskScore
# ==========================================================


ME_risk = ME_matrix.join(
    risk_df[
        [
            "RiskScore"
        ]
    ],
    how="inner"
)


print(
    "Merged data:",
    ME_risk.shape
)


ME_risk.head()

In [ ]:
# ==========================================================
# Step12.14.2
# Module-RiskScore correlation
# ==========================================================

from scipy.stats import spearmanr
import pandas as pd


module_risk_results = []


for module in ME_matrix.columns:

    rho, p = spearmanr(
        ME_risk[module],
        ME_risk["RiskScore"]
    )


    module_risk_results.append(
        {
            "Module": module,
            "Spearman_rho": rho,
            "P_value": p
        }
    )


module_risk_corr = pd.DataFrame(
    module_risk_results
)


# FDR correction
module_risk_corr["FDR"] = (
    module_risk_corr["P_value"]
    *
    len(module_risk_corr)
    /
    module_risk_corr["P_value"].rank()
)


module_risk_corr = (
    module_risk_corr
    .sort_values(
        "Spearman_rho",
        ascending=False
    )
)


module_risk_corr

In [ ]:
module_risk_corr.to_csv(
r".\LIHC_RNA_Step12_WGCNA\Results\LIHC_WGCNA_Module_RiskScore_Correlation.csv",
index=False
)

In [ ]:
# ==========================================================
# Step12.10
# Define Risk Group by median
# ==========================================================

import numpy as np
import pandas as pd


risk_cutoff = (
    risk_df["RiskScore"]
    .median()
)


risk_group = pd.DataFrame(
    index=risk_df.index
)


risk_group["RiskScore"] = (
    risk_df["RiskScore"]
)


risk_group["RiskGroup"] = np.where(
    risk_group["RiskScore"] >= risk_cutoff,
    "High",
    "Low"
)


print(
    risk_group["RiskGroup"].value_counts()
)


risk_group.head()

In [ ]:
ME_group = ME_matrix.join(
    risk_group["RiskGroup"],
    how="inner"
)


print(
    ME_group.shape
)

In [ ]:
from scipy.stats import mannwhitneyu


module_group_results=[]


for module in ME_matrix.columns:


    high = ME_group.loc[
        ME_group["RiskGroup"]=="High",
        module
    ]


    low = ME_group.loc[
        ME_group["RiskGroup"]=="Low",
        module
    ]


    stat,p = mannwhitneyu(
        high,
        low,
        alternative="two-sided"
    )


    module_group_results.append(
        {
            "Module":module,
            "High_mean":high.mean(),
            "Low_mean":low.mean(),
            "Difference":high.mean()-low.mean(),
            "P_value":p
        }
    )


module_group_df = pd.DataFrame(
    module_group_results
)


module_group_df["FDR"] = (
    module_group_df["P_value"]
    *
    len(module_group_df)
    /
    module_group_df["P_value"].rank()
)


module_group_df.sort_values(
    "P_value"
)

In [ ]:
module_group_df.to_csv(
r".\LIHC_RNA_Step12_WGCNA\Results\LIHC_WGCNA_Module_RiskGroup_Difference.csv",
index=False
)

In [ ]:
# ==========================================================
# Figure 12A
# Module-RiskScore correlation heatmap
# ==========================================================

import matplotlib.pyplot as plt
import seaborn as sns


plot_df = (
    module_risk_corr
    .set_index("Module")
    [["Spearman_rho"]]
)


plt.figure(
    figsize=(4,5)
)


sns.heatmap(
    plot_df,
    annot=True,
    cmap="RdBu_r",
    center=0,
    fmt=".2f",
    linewidths=0.5,
    cbar_kws={
        "label":"Spearman rho"
    }
)


plt.title(
    "WGCNA Module-RiskScore Correlation"
)


plt.tight_layout()


plt.savefig(
    r".\LIHC_RNA_Step12_WGCNA\Figures\Module_RiskScore_Correlation_heatmap.pdf",
    dpi=600,
    bbox_inches="tight"
)


plt.show()

In [ ]:
# ==========================================================
# Figure 12B
# Key module boxplot
# ==========================================================

import seaborn as sns
import matplotlib.pyplot as plt


key_modules = [
    "ME_blue",
    "ME_red",
    "ME_brown"
]


for module in key_modules:

    plt.figure(
        figsize=(4,4)
    )


    sns.boxplot(
        data=ME_group,
        x="RiskGroup",
        y=module
    )


    sns.stripplot(
        data=ME_group,
        x="RiskGroup",
        y=module,
        size=2,
        alpha=0.4
    )


    plt.title(
        module + 
        " by Risk Group"
    )


    plt.tight_layout()


    plt.savefig(
        rf".\LIHC_RNA_Step12_WGCNA\Figures\{module}_RiskGroup_boxplot.pdf",
        dpi=600,
        bbox_inches="tight"
    )


    plt.show()

In [ ]:
module_risk_corr.to_csv(
r".\LIHC_RNA_Step12_WGCNA\Results\LIHC_WGCNA_Module_RiskScore_Correlation.csv",
index=False
)


module_group_df.to_csv(
r".\LIHC_RNA_Step12_WGCNA\Results\LIHC_WGCNA_Module_RiskGroup_Difference.csv",
index=False
)

In [ ]:
# ==========================================================
# Step12.11.1 Check objects
# ==========================================================


print("Expression:")
print(expression_top5000.shape)


print("\nME:")
print(ME_matrix.shape)


print("\nModule assignment:")
print(module_assignment.shape)


print(module_assignment.head())

## ⚠ 已废弃的 Hub 分支：仅供历史追溯，不参与当前主分析

下方原 Cell55–91 将“Gene × 所有模块 ME”的相关对象误当成真实模块归属。
由此产生的候选、Top20、MM–GS、Hub 排名和 GO/KEGG 富集不能作为 strict-assigned 结果。

**以下 37 个单元格已改为 Raw 类型：代码文本原样保留，“运行全部”不会执行。**
原执行次数与输出保存在各 Cell 的 strict_cleanup_archive 元数据内；
完整历史另保存在修改前备份中。不要将此分支改回 Code 后直接运行。

当前主分析接续下方 strict-assigned 入口（原 Cell92）。
历史 CSV/图表未删除或覆盖，不能因名称含 Final 而作为严格版使用。


## 当前主流程：strict-assigned Hub 筛选与验证

读取既有 LIHC_Strict_Assigned_Hub_Table.csv，不重建 WGCNA、模块 eigengene 或 MM/GS。
每个基因仅使用真实 Assigned Module；MM 为与该模块 ME 的 Pearson 相关，
GS 为与 RiskScore 的 Spearman 相关，Hub_score = |MM| × |GS|。
候选阈值：|MM| ≥ 0.80、|GS| ≥ 0.20；Blue/Red/Brown 各按 Hub_score 取 Top6。
Top6 是本项目的统一验证数量，锁定名单只用于断言，不能替代动态筛选。

严格验证和正确绘图逻辑保留，末尾汇总从严格筛选对象动态生成。
本次仅编辑 Notebook，未执行；磁盘上已有的 Final_WGCNA_Hub_gene_summary.csv
仍是旧文件，只有未来明确执行修正后的汇总保存步骤才会更新。


In [ ]:
from hcc_release_paths import project_path as _hcc_path
import pandas as pd

Hub_assigned = pd.read_csv(
    _hcc_path('LIHC_RNA_Step13_Molecular_Subtyping/Input/LIHC_Strict_Assigned_Hub_Table.csv')
)

print(Hub_assigned.shape)
print(Hub_assigned.columns.tolist())

Hub_assigned.head()


# Validate the existing strict-assigned input; do not recompute MM/GS or WGCNA.
required_hub_columns = {"Gene", "Module", "MM", "GS", "Hub_score"}
assert required_hub_columns.issubset(Hub_assigned.columns), "Missing strict Hub columns"
assert Hub_assigned[["Gene", "Module"]].notna().all().all()
assert Hub_assigned["Gene"].is_unique, "Expected one row per gene"
assert not Hub_assigned["Module"].eq("grey").any()
assert module_assignment["Gene"].is_unique
_strict_assignment_check = Hub_assigned[["Gene", "Module"]].merge(
    module_assignment[["Gene", "Module"]].rename(columns={"Module": "Assigned_Module"}),
    on="Gene", how="left", validate="one_to_one", indicator=True
)
assert _strict_assignment_check["_merge"].eq("both").all(), "Unknown strict gene"
assert _strict_assignment_check["Module"].eq(
    _strict_assignment_check["Assigned_Module"]
).all(), "Strict module differs from assigned module"
assert np.isfinite(Hub_assigned[["MM", "GS", "Hub_score"]].to_numpy(dtype=float)).all()
assert np.allclose(
    Hub_assigned["Hub_score"], Hub_assigned["MM"].abs() * Hub_assigned["GS"].abs(),
    rtol=1e-10, atol=1e-12
), "Hub_score must equal abs(MM) * abs(GS)"


In [ ]:
# ==========================================================
# Unified literature-style Hub gene screening
# Blue / Red / Brown use EXACTLY the same criteria
# ==========================================================

hub_threshold_df = Hub_assigned[
    (
        Hub_assigned["Module"].isin(
            ["blue", "red", "brown"]
        )
    )
    &
    (
        Hub_assigned["MM"].abs() >= 0.80
    )
    &
    (
        Hub_assigned["GS"].abs() >= 0.20
    )
].copy()


hub_threshold_df = (
    hub_threshold_df
    .sort_values(
        ["Module", "Hub_score"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)


print(
    hub_threshold_df["Module"].value_counts()
)


display(
    hub_threshold_df[
        [
            "Gene",
            "Module",
            "MM",
            "GS",
            "Hub_score"
        ]
    ]
)

In [ ]:
# ==========================================================
# Final unified Hub selection
# Same criteria for Blue / Red / Brown
# ==========================================================

final_hub_genes = {}

for module in ["blue", "red", "brown"]:

    temp = (
        hub_threshold_df[
            hub_threshold_df["Module"] == module
        ]
        .sort_values(
            "Hub_score",
            ascending=False
        )
        .head(6)
        .copy()
    )

    final_hub_genes[module] = temp["Gene"].tolist()

    print(f"\n===== {module.upper()} =====")

    display(
        temp[
            [
                "Gene",
                "MM",
                "GS",
                "Hub_score"
            ]
        ]
    )

In [ ]:
# ==========================================================
# Step12.15.1
# Prepare Hub validation genes
# Unified selection:
# |MM| >= 0.80, |GS| >= 0.20,
# then Top 6 by Hub_score within each module
# ==========================================================


Blue_validation_genes = [
    "CENPA",
    "KIF2C",
    "CCNB1",
    "CDC20",
    "PRR11",
    "CDCA8"
]


Brown_validation_genes = [
    "CYP8B1",
    "GLYATL1",
    "PLG",
    "ACSM2A",
    "ABAT",
    "MAT1A"
]


Red_validation_genes = [
    "SLC52A2",
    "RPL8",
    "BOP1",
    "ZFAS1",
    "SNHG6",
    "RPL7P9"
]


Hub_validation_genes = (
    Blue_validation_genes
    +
    Brown_validation_genes
    +
    Red_validation_genes
)


len(Hub_validation_genes)


# Locked lists are assertion targets, never the source of dynamic selection.
_strict_expected_hub18 = {
    "blue": Blue_validation_genes,
    "red": Red_validation_genes,
    "brown": Brown_validation_genes,
}
assert set(final_hub_genes) == set(_strict_expected_hub18)
for _module, _expected_genes in _strict_expected_hub18.items():
    _selected_genes = final_hub_genes[_module]
    assert len(_selected_genes) == 6 and len(set(_selected_genes)) == 6
    assert set(_selected_genes) == set(_expected_genes), (
        f"{_module}: Top6 differs from locked strict Hub18; stop, do not replace"
    )
assert len({g for gs in final_hub_genes.values() for g in gs}) == 18


In [ ]:
expected_module = {
    **{g: "blue" for g in Blue_validation_genes},
    **{g: "brown" for g in Brown_validation_genes},
    **{g: "red" for g in Red_validation_genes}
}

validation_check = module_assignment[
    module_assignment["Gene"].isin(Hub_validation_genes)
].copy()

validation_check["Expected_Module"] = (
    validation_check["Gene"].map(expected_module)
)

validation_check["Correct"] = (
    validation_check["Module"]
    ==
    validation_check["Expected_Module"]
)

display(
    validation_check.sort_values(
        ["Expected_Module", "Gene"]
    )
)

print(
    "All module assignments correct:",
    validation_check["Correct"].all()
)

In [ ]:
# ==========================================================
# Extract hub expression
# ==========================================================


Hub_expression = (
    expression_top5000
    .loc[
        Hub_validation_genes
    ]
    .T
)


print(
    Hub_expression.shape
)


Hub_expression.head()

In [ ]:
Hub_survival_df = Hub_expression.join(
    risk_df[
        [
            "RiskScore",
            "OS_time",
            "OS"
        ]
    ],
    how="inner"
)


print(
    Hub_survival_df.shape
)

In [ ]:
Hub_survival_df = Hub_expression.join(
    risk_df[
        [
            "RiskScore",
            "OS_time",
            "OS"
        ]
    ],
    how="inner"
)


print(
    Hub_survival_df.shape
)

In [ ]:
# ==========================================================
# Step12.15.2
# Hub gene - RiskScore correlation
# ==========================================================

from scipy.stats import spearmanr
import pandas as pd


corr_results=[]


for gene in Hub_validation_genes:

    rho,p = spearmanr(
        Hub_survival_df[gene],
        Hub_survival_df["RiskScore"]
    )


    corr_results.append(
        [
            gene,
            rho,
            p
        ]
    )


Hub_Risk_Corr = pd.DataFrame(
    corr_results,
    columns=[
        "Gene",
        "Spearman_rho",
        "P_value"
    ]
)


# FDR校正

from statsmodels.stats.multitest import multipletests


Hub_Risk_Corr["FDR"] = (
    multipletests(
        Hub_Risk_Corr["P_value"],
        method="fdr_bh"
    )[1]
)


Hub_Risk_Corr = (
    Hub_Risk_Corr
    .sort_values(
        "Spearman_rho",
        ascending=False
    )
)


Hub_Risk_Corr

In [ ]:
Hub_Risk_Corr.to_csv(
    r".\LIHC_RNA_Step12_WGCNA\Results\HubGene_RiskScore_Correlation.csv",
    index=False
)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


plot_df = (
    Hub_Risk_Corr
    .set_index("Gene")
    [["Spearman_rho"]]
)


plt.figure(
    figsize=(3,6)
)


sns.heatmap(
    plot_df,
    annot=True,
    cmap="coolwarm",
    center=0,
    linewidths=0.5
)


plt.title(
    "Hub Gene Correlation with RiskScore"
)


plt.tight_layout()


plt.savefig(
    r".\LIHC_RNA_Step12_WGCNA\Figures\Hub_RiskScore_Correlation_heatmap.pdf",
    dpi=600,
    bbox_inches="tight"
)


plt.show()

In [ ]:
# ==========================================================
# Step12.15.3
# Hub gene Kaplan-Meier
# ==========================================================


import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test
import numpy as np


def hub_gene_KM(
        df,
        gene
):

    # median cutoff

    cutoff = df[gene].median()


    high = df[
        df[gene] >= cutoff
    ]


    low = df[
        df[gene] < cutoff
    ]


    kmf_high = KaplanMeierFitter()
    kmf_low = KaplanMeierFitter()


    plt.figure(
        figsize=(5,4)
    )


    kmf_high.fit(
        high["OS_time"],
        high["OS"],
        label="High"
    )


    kmf_low.fit(
        low["OS_time"],
        low["OS"],
        label="Low"
    )


    kmf_high.plot()
    kmf_low.plot()


    result = logrank_test(
        high["OS_time"],
        low["OS_time"],
        event_observed_A=high["OS"],
        event_observed_B=low["OS"]
    )


    plt.title(
        f"{gene} survival\np={result.p_value:.3e}"
    )


    plt.xlabel(
        "Time (days)"
    )


    plt.ylabel(
        "Overall survival"
    )


    plt.tight_layout()


    plt.show()


    return result.p_value

In [ ]:
risk_genes = [
    # Blue risk module
    "CENPA",
    "KIF2C",
    "CCNB1",
    "CDC20",
    "PRR11",
    "CDCA8",

    # Red risk module
    "SLC52A2",
    "RPL8",
    "BOP1",
    "ZFAS1",
    "SNHG6",
    "RPL7P9"
]

In [ ]:
protective_genes = [
    "CYP8B1",
    "GLYATL1",
    "PLG",
    "ACSM2A",
    "ABAT",
    "MAT1A"
]

In [ ]:
Hub_genes_18 = (
    risk_genes
    +
    protective_genes
)


Hub_expression_18 = (
    expression_top5000
    .loc[
        Hub_genes_18
    ]
    .T
)


print(
    Hub_expression_18.shape
)

In [ ]:
Hub_survival_df_18 = Hub_expression_18.join(
    risk_df[
        [
            "RiskScore",
            "OS_time",
            "OS"
        ]
    ],
    how="inner"
)


print(
    Hub_survival_df_18.shape
)

In [ ]:
for gene in risk_genes:
    print(
        gene,
        hub_gene_KM(
            Hub_survival_df_18,
            gene
        )
    )

In [ ]:
Hub_validation_genes = (
    risk_genes
    +
    protective_genes
)

In [ ]:
# ==========================================================
# Step12.15.3
# Hub gene KM survival analysis (18 genes)
# ==========================================================


import os
import matplotlib.pyplot as plt

from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test


km_dir = r".\LIHC_RNA_Step12_WGCNA\Figures\Hub_KM"

os.makedirs(
    km_dir,
    exist_ok=True
)



def plot_gene_KM(
        df,
        gene
):

    # median cutoff

    cutoff = df[gene].median()


    high_group = df[
        df[gene] >= cutoff
    ]


    low_group = df[
        df[gene] < cutoff
    ]


    km_high = KaplanMeierFitter()
    km_low = KaplanMeierFitter()


    plt.figure(
        figsize=(5,4)
    )


    km_high.fit(
        high_group["OS_time"],
        high_group["OS"],
        label=f"{gene} High"
    )


    km_low.fit(
        low_group["OS_time"],
        low_group["OS"],
        label=f"{gene} Low"
    )


    ax = plt.gca()


    km_high.plot(
        ax=ax,
        linewidth=2
    )


    km_low.plot(
        ax=ax,
        linewidth=2
    )


    # log-rank test

    lr = logrank_test(
        high_group["OS_time"],
        low_group["OS_time"],
        event_observed_A=high_group["OS"],
        event_observed_B=low_group["OS"]
    )


    plt.title(
        f"{gene}\nLog-rank P={lr.p_value:.3e}"
    )


    plt.xlabel(
        "Time (days)"
    )


    plt.ylabel(
        "Overall survival"
    )


    plt.legend()


    plt.tight_layout()


    plt.savefig(
        os.path.join(
            km_dir,
            f"{gene}_KM.png"
        ),
        dpi=600,
        bbox_inches="tight"
    )


    plt.savefig(
        os.path.join(
            km_dir,
            f"{gene}_KM.pdf"
        ),
        dpi=600,
        bbox_inches="tight"
    )


    plt.close()


    return {
        "Gene":gene,
        "Cutoff":cutoff,
        "P_value":lr.p_value,
        "High_n":len(high_group),
        "Low_n":len(low_group)
    }

In [ ]:
# ==========================================================
# 18 Hub genes from Blue + Red + Brown modules
# Unified final version
# ==========================================================

Hub_genes_18 = [

    # Blue risk module
    "CENPA",
    "KIF2C",
    "CCNB1",
    "CDC20",
    "PRR11",
    "CDCA8",

    # Brown protective module
    "CYP8B1",
    "GLYATL1",
    "PLG",
    "ACSM2A",
    "ABAT",
    "MAT1A",

    # Red risk module
    "SLC52A2",
    "RPL8",
    "BOP1",
    "ZFAS1",
    "SNHG6",
    "RPL7P9"
]

len(Hub_genes_18)

In [ ]:
# ==========================================================
# Batch KM
# ==========================================================


KM_results=[]


for gene in Hub_genes_18:

    result = plot_gene_KM(
        Hub_survival_df,
        gene
    )

    KM_results.append(
        result
    )


KM_results = pd.DataFrame(
    KM_results
)


KM_results

In [ ]:
KM_results.to_csv(
    r".\LIHC_RNA_Step12_WGCNA\Results\Hub_gene_KM_results.csv",
    index=False
)

In [ ]:
# ==========================================================
# Step12.15.3 Visualization
# Hub gene KM summary plot (18 Hub genes)
# ==========================================================

import matplotlib.pyplot as plt
import numpy as np


KM_plot = KM_results.copy()


# -log10 P
KM_plot["logP"] = (
    -np.log10(
        KM_plot["P_value"]
    )
)


# ==========================================================
# Module classification
# ==========================================================

Blue_genes = [
    "CENPA","KIF2C","CCNB1",
    "CDC20","PRR11","CDCA8"
]

Red_genes = [
    "SLC52A2","RPL8","BOP1",
    "ZFAS1","SNHG6","RPL7P9"
]

Brown_genes = [
    "CYP8B1","GLYATL1","PLG",
    "ACSM2A","ABAT","MAT1A"
]


def classify_module(gene):

    if gene in Blue_genes:
        return "Blue_CellCycle_Risk"

    elif gene in Red_genes:
        return "Red_TumorProgression_Risk"

    elif gene in Brown_genes:
        return "Brown_Metabolism_Protective"

    else:
        return "Other"


KM_plot["Module"] = (
    KM_plot["Gene"]
    .apply(classify_module)
)


# 排序
KM_plot = KM_plot.sort_values(
    "logP"
)


# ==========================================================
# Plot
# ==========================================================

plt.figure(
    figsize=(8,6)
)


for module in KM_plot["Module"].unique():

    temp = KM_plot[
        KM_plot["Module"] == module
    ]

    plt.scatter(
        temp["logP"],
        temp["Gene"],
        s=140,
        label=module
    )


plt.xlabel(
    "-log10(Log-rank P)"
)

plt.ylabel(
    ""
)

plt.title(
    "Survival Validation of WGCNA Hub Genes"
)

plt.legend(
    bbox_to_anchor=(1.05,1),
    loc="upper left"
)

plt.tight_layout()


plt.savefig(
    r".\LIHC_RNA_Step12_WGCNA\Figures\Hub_gene18_KM_summary.pdf",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    r".\LIHC_RNA_Step12_WGCNA\Figures\Hub_gene18_KM_summary.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()
plt.close()

In [ ]:
import seaborn as sns


heat_df = (
    KM_results
    .set_index("Gene")
    [["P_value"]]
)


heat_df["-log10P"] = (
    -np.log10(
        heat_df["P_value"]
    )
)


heat_df = heat_df[
    ["-log10P"]
]


plt.figure(
    figsize=(3,6)
)


sns.heatmap(
    heat_df,
    annot=True,
    cmap="Reds",
    linewidths=0.5
)


plt.title(
    "Hub gene survival significance"
)


plt.tight_layout()


plt.savefig(
    r".\LIHC_RNA_Step12_WGCNA\Figures\Hub_gene_KM_heatmap.pdf",
    dpi=600,
    bbox_inches="tight"
)


plt.show()

In [ ]:
# ==========================================================
# Rebuild corrected 18-gene survival dataframe
# ==========================================================

Hub_expression_18 = (
    expression_top5000
    .loc[Hub_genes_18]
    .T
)

print("Hub_expression_18:", Hub_expression_18.shape)


Hub_survival_df_18 = (
    Hub_expression_18
    .join(
        risk_df[
            [
                "RiskScore",
                "OS_time",
                "OS"
            ]
        ],
        how="inner"
    )
)


print(
    "Hub_survival_df_18:",
    Hub_survival_df_18.shape
)

print(
    Hub_survival_df_18.columns.tolist()
)

In [ ]:
# ==========================================================
# Step12.15.4
# Univariate Cox regression
# 18 Hub genes
# ==========================================================

from lifelines import CoxPHFitter
import pandas as pd


cox_results = []


for gene in Hub_genes_18:

    cox_df = Hub_survival_df_18[
        [
            gene,
            "OS_time",
            "OS"
        ]
    ].copy()


    cph = CoxPHFitter()


    cph.fit(
        cox_df,
        duration_col="OS_time",
        event_col="OS"
    )


    result = cph.summary.loc[gene]


    cox_results.append(
        [
            gene,
            result["exp(coef)"],
            result["exp(coef) lower 95%"],
            result["exp(coef) upper 95%"],
            result["p"]
        ]
    )



Hub_Cox18 = pd.DataFrame(
    cox_results,
    columns=[
        "Gene",
        "HR",
        "CI_low",
        "CI_high",
        "P_value"
    ]
)



Hub_Cox18

In [ ]:
# ==========================================================
# Add module annotation
# ==========================================================


def get_module(gene):

    if gene in Blue_genes:
        return "Blue_CellCycle"

    elif gene in Red_genes:
        return "Red_Progression"

    elif gene in Brown_genes:
        return "Brown_Metabolism"

    else:
        return "Other"



Hub_Cox18["Module"] = (
    Hub_Cox18["Gene"]
    .apply(get_module)
)


Hub_Cox18

In [ ]:
Hub_Cox18.to_csv(
    r".\LIHC_RNA_Step12_WGCNA\Results\Hub_gene18_Cox_results.csv",
    index=False
)

In [ ]:
# ==========================================================
# Cox forest plot
# ==========================================================

import matplotlib.pyplot as plt
import numpy as np


forest_df = Hub_Cox18.copy()


forest_df = forest_df.sort_values(
    "HR"
)



plt.figure(
    figsize=(8,8)
)


y = np.arange(
    len(forest_df)
)


plt.errorbar(
    forest_df["HR"],
    y,
    xerr=[
        forest_df["HR"]-forest_df["CI_low"],
        forest_df["CI_high"]-forest_df["HR"]
    ],
    fmt="o"
)


plt.axvline(
    1,
    linestyle="--"
)



plt.yticks(
    y,
    forest_df["Gene"]
)



plt.xlabel(
    "Hazard Ratio (95% CI)"
)



plt.title(
    "Univariate Cox Regression of WGCNA Hub Genes"
)



plt.tight_layout()



plt.savefig(
    r".\LIHC_RNA_Step12_WGCNA\Figures\Hub_gene18_Cox_forest.pdf",
    dpi=600,
    bbox_inches="tight"
)


plt.savefig(
    r".\LIHC_RNA_Step12_WGCNA\Figures\Hub_gene18_Cox_forest.png",
    dpi=600,
    bbox_inches="tight"
)


plt.show()

In [ ]:
immune_df = pd.read_csv(
    r".\LIHC_RNA_Step11_Immune\Results\LIHC_Integrated_Immune_Correlation_Matrix.csv",
    index_col=0
)


print(immune_df.shape)

immune_df.head()

In [ ]:
# ==========================================================
# Step12.16.1
# Load immune infiltration matrix
# ==========================================================


immune_matrix = pd.read_csv(
    r".\LIHC_RNA_Step11_Immune\Results\LIHC_Integrated_Immune_Matrix_365.csv",
    index_col=0
)


print(
    immune_matrix.shape
)


immune_matrix.head()

In [ ]:
ME_core = ME_matrix[
    [
        "ME_blue",
        "ME_red",
        "ME_brown"
    ]
]


print(
    ME_core.shape
)

In [ ]:
ME_immune = ME_core.join(
    immune_matrix,
    how="inner"
)


print(
    ME_immune.shape
)

In [ ]:
from scipy.stats import spearmanr


module_names = [
    "ME_blue",
    "ME_red",
    "ME_brown"
]


immune_names = [
    col for col in immune_matrix.columns
]


corr_matrix = pd.DataFrame(
    index=module_names,
    columns=immune_names
)


p_matrix = pd.DataFrame(
    index=module_names,
    columns=immune_names
)



for m in module_names:

    for immune in immune_names:

        rho,p = spearmanr(
            ME_immune[m],
            ME_immune[immune]
        )


        corr_matrix.loc[m,immune]=rho

        p_matrix.loc[m,immune]=p



corr_matrix = corr_matrix.astype(float)

p_matrix = p_matrix.astype(float)



corr_matrix.head()

In [ ]:
# ==========================================================
# Step12.16.1
# Load immune matrix
# ==========================================================

immune_matrix = pd.read_csv(
    r".\LIHC_RNA_Step11_Immune\Results\LIHC_Integrated_Immune_Matrix_365.csv",
    index_col=0
)


print(immune_matrix.shape)

immune_matrix.head()

In [ ]:
# ==========================================================
# Core WGCNA modules
# ==========================================================


ME_core = ME_matrix[
    [
        "ME_blue",
        "ME_red",
        "ME_brown"
    ]
]


print(
    ME_core.shape
)


ME_core.head()

In [ ]:
# ==========================================================
# Merge WGCNA and immune
# ==========================================================


ME_immune = ME_core.join(
    immune_matrix,
    how="inner"
)


print(
    ME_immune.shape
)

In [ ]:
# ==========================================================
# Spearman correlation
# ==========================================================

from scipy.stats import spearmanr
import pandas as pd
import numpy as np


modules = [
    "ME_blue",
    "ME_red",
    "ME_brown"
]


immune_features = [
    x for x in immune_matrix.columns
]


corr_matrix = pd.DataFrame(
    index=modules,
    columns=immune_features
)


p_matrix = pd.DataFrame(
    index=modules,
    columns=immune_features
)



for module in modules:

    for immune in immune_features:


        rho,p = spearmanr(
            ME_immune[module],
            ME_immune[immune]
        )


        corr_matrix.loc[module,immune]=rho

        p_matrix.loc[module,immune]=p



corr_matrix = corr_matrix.astype(float)

p_matrix = p_matrix.astype(float)



print(
    corr_matrix.shape
)


corr_matrix

In [ ]:
from statsmodels.stats.multitest import multipletests


p_values = (
    p_matrix
    .values
    .flatten()
)


fdr_values = multipletests(
    p_values,
    method="fdr_bh"
)[1]


fdr_matrix = pd.DataFrame(
    fdr_values.reshape(
        p_matrix.shape
    ),
    index=p_matrix.index,
    columns=p_matrix.columns
)


fdr_matrix.head()

In [ ]:
corr_matrix.to_csv(
    r".\LIHC_RNA_Step12_WGCNA\Results\WGCNA_Immune_Module_Correlation.csv"
)


fdr_matrix.to_csv(
    r".\LIHC_RNA_Step12_WGCNA\Results\WGCNA_Immune_Module_FDR.csv"
)

In [ ]:
# ==========================================================
# WGCNA module immune correlation heatmap
# ==========================================================


import seaborn as sns
import matplotlib.pyplot as plt


immune_selected = [

    # cells
    "Cell_CD8_T_cells",
    "Cell_Cytotoxic_cells",
    "Cell_NK_cells",
    "Cell_M1_macrophages",
    "Cell_M2_macrophages",
    "Cell_Macrophages",

    # functions
    "Function_T_cell_activation",
    "Function_T_cell_exhaustion",
    "Function_Type_I_IFN_response",
    "Function_Type_II_IFN_response",

    # checkpoints
    "Checkpoint_CD276",
    "Checkpoint_TNFRSF4",
    "Checkpoint_VTCN1",
    "Checkpoint_CD40LG",
    "Checkpoint_CD160"

]


corr_heatmap = corr_matrix[
    immune_selected
]


plt.figure(
    figsize=(12,3)
)


sns.heatmap(
    corr_heatmap,
    cmap="RdBu_r",
    center=0,
    annot=True,
    fmt=".2f",
    linewidths=0.5
)


plt.title(
    "WGCNA Module-Immune Correlation"
)


plt.tight_layout()


plt.savefig(
    r".\LIHC_RNA_Step12_WGCNA\Figures\Module_Immune_heatmap.pdf",
    dpi=600,
    bbox_inches="tight"
)


plt.savefig(
    r".\LIHC_RNA_Step12_WGCNA\Figures\Module_Immune_heatmap.png",
    dpi=600,
    bbox_inches="tight"
)


plt.show()

In [ ]:
# ==========================================================
# Optimized WGCNA Module-Immune Heatmap
# ==========================================================

import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np


# --------------------------
# 选择指标
# --------------------------

immune_selected = [

    # immune cells
    "Cell_CD8_T_cells",
    "Cell_Cytotoxic_cells",
    "Cell_NK_cells",
    "Cell_M1_macrophages",
    "Cell_M2_macrophages",
    "Cell_Macrophages",

    # immune function
    "Function_T_cell_activation",
    "Function_T_cell_exhaustion",
    "Function_Type_I_IFN_response",
    "Function_Type_II_IFN_response",

    # checkpoint
    "Checkpoint_CD276",
    "Checkpoint_TNFRSF4",
    "Checkpoint_VTCN1",
    "Checkpoint_CD40LG",
    "Checkpoint_CD160"

]


heat_df = corr_matrix[
    immune_selected
].copy()



# --------------------------
# 修改标签
# --------------------------

rename_dict = {

    "Cell_CD8_T_cells":"CD8 T",
    "Cell_Cytotoxic_cells":"Cytotoxic",
    "Cell_NK_cells":"NK",
    "Cell_M1_macrophages":"M1 Macro",
    "Cell_M2_macrophages":"M2 Macro",
    "Cell_Macrophages":"Macrophage",

    "Function_T_cell_activation":"T activation",
    "Function_T_cell_exhaustion":"T exhaustion",
    "Function_Type_I_IFN_response":"Type I IFN",
    "Function_Type_II_IFN_response":"Type II IFN",

    "Checkpoint_CD276":"CD276",
    "Checkpoint_TNFRSF4":"TNFRSF4",
    "Checkpoint_VTCN1":"VTCN1",
    "Checkpoint_CD40LG":"CD40LG",
    "Checkpoint_CD160":"CD160"

}


heat_df.rename(
    columns=rename_dict,
    inplace=True
)



# --------------------------
# 模块名字优化
# --------------------------

heat_df.index = [

    "Blue\n(Cell cycle)",
    "Red\n(Progression)",
    "Brown\n(Metabolism)"

]



# --------------------------
# 绘图
# --------------------------

plt.figure(
    figsize=(12,4)
)


sns.set_style(
    "white"
)


ax = sns.heatmap(

    heat_df,

    cmap="RdBu_r",

    center=0,

    vmin=-0.6,
    vmax=0.6,

    linewidths=0.8,

    linecolor="white",

    annot=True,

    fmt=".2f",

    annot_kws={
        "size":8
    },

    cbar_kws={
        "label":"Spearman ρ"
    }

)



plt.xticks(

    rotation=45,

    ha="right",

    fontsize=10

)


plt.yticks(

    rotation=0,

    fontsize=11

)


plt.title(

    "Correlation Between WGCNA Modules and Immune Features",

    fontsize=14,

    pad=15

)



plt.tight_layout()



plt.savefig(

    r".\LIHC_RNA_Step12_WGCNA\Figures\WGCNA_Immune_heatmap_final.pdf",

    dpi=600,

    bbox_inches="tight"

)


plt.savefig(

    r".\LIHC_RNA_Step12_WGCNA\Figures\WGCNA_Immune_heatmap_final.png",

    dpi=600,

    bbox_inches="tight"

)


plt.show()

In [ ]:
# ==========================================================
# Step12.17.1
# Module Risk correlation summary
# Recalculate from current ME_matrix
# ==========================================================

from scipy.stats import spearmanr
import pandas as pd


common_samples = ME_matrix.index.intersection(
    risk_df.index
)


module_info = {
    "ME_blue": "Cell cycle",
    "ME_red": "Tumor progression",
    "ME_brown": "Metabolism"
}


results = []


for module, function in module_info.items():

    rho, p = spearmanr(
        ME_matrix.loc[common_samples, module],
        risk_df.loc[common_samples, "RiskScore"]
    )

    results.append({
        "Module": module,
        "Function": function,
        "Correlation": rho,
        "P_value": p,
        "Direction": "Risk" if rho > 0 else "Protective"
    })


module_risk_summary = pd.DataFrame(results)


module_risk_summary

In [ ]:
module_risk_summary.to_csv(

r".\LIHC_RNA_Step12_WGCNA\Results\Module_Risk_Correlation_summary.csv",

index=False

)

In [ ]:
# ==========================================================
# Rebuild ME_risk using current ME_matrix
# ==========================================================

common_samples = ME_matrix.index.intersection(
    risk_df.index
)

ME_risk = (
    ME_matrix.loc[
        common_samples,
        [
            "ME_blue",
            "ME_red",
            "ME_brown"
        ]
    ]
    .join(
        risk_df.loc[
            common_samples,
            ["RiskScore"]
        ],
        how="inner"
    )
)

print(ME_risk.shape)

In [ ]:
# ==========================================================
# Module eigengene vs RiskScore
# ==========================================================

import matplotlib.pyplot as plt
import seaborn as sns


plot_df = ME_risk.copy()


modules=[
    "ME_blue",
    "ME_red",
    "ME_brown"
]


fig,axes = plt.subplots(
    1,
    3,
    figsize=(12,3.5)
)



for ax,module in zip(
    axes,
    modules
):

    sns.regplot(

        data=plot_df,

        x=module,

        y="RiskScore",

        scatter_kws={
            "s":25,
            "alpha":0.6
        },

        line_kws={
            "linewidth":2
        },

        ax=ax

    )


    ax.set_title(
        module
    )


    ax.set_xlabel(
        "Module Eigengene"
    )


    ax.set_ylabel(
        "Risk Score"
    )


plt.tight_layout()



plt.savefig(

r".\LIHC_RNA_Step12_WGCNA\Figures\Module_Risk_relationship.pdf",

dpi=600,

bbox_inches="tight"

)


plt.show()

In [ ]:
# Self-contained strict-assigned Hub summary; no files are written.
import pandas as pd
import numpy as np

_summary_input_path = (
    r"E:\TCGA\LIHC_RNA_Step13_Molecular_Subtyping\Input"
    r"\LIHC_Strict_Assigned_Hub_Table.csv"
)
_summary_strict_table = pd.read_csv(_summary_input_path)
_summary_required_columns = {"Gene", "Module", "MM", "GS", "Hub_score"}
assert _summary_required_columns.issubset(_summary_strict_table.columns), (
    "Missing columns: "
    + str(sorted(_summary_required_columns - set(_summary_strict_table.columns)))
)
assert _summary_strict_table[["Gene", "Module"]].notna().all().all(), (
    "Missing gene or assigned module"
)
assert _summary_strict_table["Gene"].is_unique, (
    "Strict assigned table must contain one record per gene"
)

# Normalize module labels in this local copy only.
_summary_strict_table["Module"] = (
    _summary_strict_table["Module"].astype(str).str.strip().str.lower()
)
_summary_key_table = _summary_strict_table.loc[
    _summary_strict_table["Module"].isin(["blue", "red", "brown"])
].copy()
for _summary_column in ["MM", "GS", "Hub_score"]:
    _summary_key_table[_summary_column] = pd.to_numeric(
        _summary_key_table[_summary_column], errors="raise"
    )
assert np.isfinite(
    _summary_key_table[["MM", "GS", "Hub_score"]].to_numpy(dtype=float)
).all(), "Non-finite strict Hub statistics"
assert np.allclose(
    _summary_key_table["Hub_score"],
    _summary_key_table["MM"].abs() * _summary_key_table["GS"].abs(),
    rtol=1e-10, atol=1e-12
), "Hub_score must equal abs(MM) * abs(GS)"

_summary_candidates = _summary_key_table.loc[
    (_summary_key_table["MM"].abs() >= 0.80)
    & (_summary_key_table["GS"].abs() >= 0.20)
].copy()

# Locked genes are assertion targets ONLY, never selection or output data.
_summary_locked_hub18 = {
    "blue": ["CENPA", "KIF2C", "CCNB1", "CDC20", "PRR11", "CDCA8"],
    "red": ["SLC52A2", "RPL8", "BOP1", "ZFAS1", "SNHG6", "RPL7P9"],
    "brown": ["CYP8B1", "GLYATL1", "PLG", "ACSM2A", "ABAT", "MAT1A"],
}
_summary_function_labels = {
    "blue": "Cell cycle",
    "red": "Tumor progression",
    "brown": "Metabolism",
}
_summary_parts = []
for _summary_module, _summary_function in _summary_function_labels.items():
    _summary_top6 = (
        _summary_candidates.loc[
            _summary_candidates["Module"].eq(_summary_module)
        ]
        .sort_values("Hub_score", ascending=False)
        .head(6)
        .copy()
    )
    assert len(_summary_top6) == 6 and _summary_top6["Gene"].is_unique, (
        f"{_summary_module}: expected six unique qualifying genes"
    )
    assert set(_summary_top6["Gene"]) == set(
        _summary_locked_hub18[_summary_module]
    ), f"{_summary_module}: dynamic Top6 differs from locked strict Hub18"
    _summary_part = _summary_top6[["Gene", "Module"]].copy()
    _summary_part["Module"] = _summary_part["Module"].str.capitalize()
    _summary_part["Biological_Function"] = _summary_function
    _summary_parts.append(_summary_part)

_summary_result = pd.concat(_summary_parts, ignore_index=True)
assert len(_summary_result) == 18 and _summary_result["Gene"].is_unique
assert _summary_result["Module"].value_counts().to_dict() == {
    "Blue": 6, "Red": 6, "Brown": 6
}
hub_summary = _summary_result
print(hub_summary[["Gene", "Module"]].to_string(index=False))
print("PASS: all 18 genes and assigned modules match the locked strict Hub18.")


In [ ]:
hub_summary.to_csv(

r".\LIHC_RNA_Step12_WGCNA\Results\Final_WGCNA_Hub_gene_summary.csv",

index=False

)